## 0 · 載入 + 工具

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 自動找 data 資料夾
CANDS = [
    '../../workshop_build/workshop_build/09_advanced_ml/gan/data',
    'workshop_build/workshop_build/09_advanced_ml/gan/data',
    'data', '../data', '../../gan/data',
]
DATA = None
for guess in CANDS:
    if os.path.exists(os.path.join(guess, 'stakeholder_relations.parquet')):
        DATA = guess; break
print('資料夾：', DATA)

df = pd.read_parquet(os.path.join(DATA, 'stakeholder_relations.parquet'))
print('資料形狀：', df.shape, '(列 = 街區, 欄 = 特徵)')
df.head()

In [ ]:
# 讓 matplotlib 顯示中文（找不到字型就略過，不影響程式）
from matplotlib import font_manager as fm
for fp in [
    '../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    '../../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    'C:/Windows/Fonts/msjh.ttc',
]:
    if os.path.exists(fp):
        try:
            fm.fontManager.addfont(fp)
            plt.rcParams['font.sans-serif'] = [fm.FontProperties(fname=fp).get_name()]
            break
        except Exception: pass
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 選特徵欄 + 標準化（ML 三本都要的前處理）
drop = ['site', 'gx', 'gy']
feat_cols = [c for c in df.columns
             if c not in drop and np.issubdtype(df[c].dtype, np.number)]
X = df[feat_cols].to_numpy(dtype='float32')
mu = X.mean(axis=0); sd = X.std(axis=0); sd[sd < 1e-8] = 1.0
Xs = (X - mu) / sd

share_cols = ['share_state','share_developer','share_resident','share_unknown']
share_idx  = [feat_cols.index(c) for c in share_cols]
names  = ['state 政府','developer 開發商','resident 居民','unknown 未知']
colors = ['#1f77b4','#d62728','#2ca02c','#7f7f7f']
dominant = X[:, share_idx].argmax(axis=1)
print('特徵欄 %d 個，標準化後 Xs =' % len(feat_cols), Xs.shape)

## 1 · 直覺：什麼叫「降維」？

想像每個街區有 13 個數字 → 它是 **13 維空間裡的一個點**，看不見。
降維 = 把它**投影**到 2 維紙面，讓我們**看得到**、又盡量不失真。

下圖：把 3 維資料（左）壓成 2 維（右）的示意。

In [ ]:
rng = np.random.default_rng(0)
n = 300
t = rng.normal(size=n)
toy3 = np.stack([t, 0.5*t + 0.3*rng.normal(size=n), -0.8*t + 0.3*rng.normal(size=n)], axis=1)

fig = plt.figure(figsize=(11,4))
ax = fig.add_subplot(1,2,1, projection='3d')
ax.scatter(toy3[:,0], toy3[:,1], toy3[:,2], s=8, c=t, cmap='viridis')
ax.set_title('原始：3 維（難看懂）')
from sklearn.decomposition import PCA
toy2 = PCA(n_components=2).fit_transform(toy3)
ax2 = fig.add_subplot(1,2,2)
ax2.scatter(toy2[:,0], toy2[:,1], s=8, c=t, cmap='viridis')
ax2.set_title('PCA 壓成 2 維（看得懂了）'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2 · PCA：找「最分散」的方向

PCA 做的事：在資料裡找一條**讓點散得最開**的直線（PC1），再找第二條垂直的（PC2），
把每個點投影到這兩條線上 → 得到 2 個座標。**只能用直線，不會彎。**

下圖用玩具 2 維資料，畫出 PCA 找到的兩個主方向（箭頭）。

In [ ]:
from sklearn.decomposition import PCA
pts = rng.normal(size=(400,2)) @ np.array([[2.0,0.8],[0.0,0.6]])
p = PCA(n_components=2).fit(pts)
center = pts.mean(0)

plt.figure(figsize=(6,6))
plt.scatter(pts[:,0], pts[:,1], s=10, alpha=0.4, color='#888')
for k in range(2):
    vec = p.components_[k] * np.sqrt(p.explained_variance_[k]) * 2.5
    plt.arrow(center[0], center[1], vec[0], vec[1], width=0.05,
              color=['#d62728','#1f77b4'][k], length_includes_head=True,
              label=f'PC{k+1}（抓 {p.explained_variance_ratio_[k]*100:.0f}% 資訊）')
plt.legend(); plt.axis('equal'); plt.grid(alpha=0.3)
plt.title('PCA：紅=最分散方向 PC1，藍=次方向 PC2'); plt.show()

**重點**：PC1 抓最多資訊，PC2 次之。`explained_variance_ratio_` 告訴你每個方向抓住多少 %。


In [ ]:
# 真實資料上的 PCA（13 維  2 維）
pca = PCA(n_components=2); Zp = pca.fit_transform(Xs)
plt.figure(figsize=(7,5.5))
for i in range(4):
    m = (dominant == i)
    plt.scatter(Zp[m,0], Zp[m,1], s=8, c=colors[i], label=names[i], alpha=0.6)
plt.title(f'真實街區 PCA（2 方向共抓 {pca.explained_variance_ratio_.sum()*100:.0f}% 資訊）')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 3 · AE 自編碼器：讓網路自己學壓法

PCA 只能直線。**AE** 用神經網路：中間掐一個「**瓶頸**」只有 2 個神經元，
強迫它把 13 個數字擠進 2 個、再還原回 13 個。為了還原得準，那 2 個數字就學成最有用的壓縮。

```
13 維 →[encoder]→ 2 維(瓶頸 latent) →[decoder]→ 13 維
        壓縮                              還原
```
因為中間有 ReLU 會彎，**AE 的 2 維圖通常比 PCA 更會把不同群分開**。

In [ ]:
import matplotlib.patches as mp
layers = [13,16,2,16,13]; labels=['輸入\n13','','瓶頸\n2','','輸出\n13']
plt.figure(figsize=(9,4)); ax=plt.gca()
for li,(nser) in enumerate(layers):
    xs = li*2.2
    ys = np.linspace(-nser/2, nser/2, nser)
    col = '#d62728' if nser==2 else '#4C72B0'
    ax.scatter([xs]*nser, ys, s=120, color=col, zorder=3)
    if labels[li]: ax.text(xs, max(ys)+1.2, labels[li], ha='center', fontsize=11)
ax.text(2.2,-9,'encoder（壓縮）',ha='center',color='#555')
ax.text(6.6,-9,'decoder（還原）',ha='center',color='#555')
ax.set_title('AE 結構：中間瓶頸只有 2 個 → 被迫學會精華壓縮')
ax.axis('off'); ax.set_ylim(-10,9); plt.show()

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0); D=Xs.shape[1]
class AE(nn.Module):
    def __init__(s):
        super().__init__()
        s.enc=nn.Sequential(nn.Linear(D,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,2))
        s.dec=nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,32),nn.ReLU(),nn.Linear(32,D))
    def forward(s,x): z=s.enc(x); return s.dec(z), z
ae=AE(); xt=torch.tensor(Xs); opt=torch.optim.Adam(ae.parameters(),1e-3); lf=nn.MSELoss()
for e in range(60):
    xh,z=ae(xt); l=lf(xh,xt); opt.zero_grad(); l.backward(); opt.step()
print('AE 還原誤差：', round(l.item(),4))
ae.eval()
with torch.no_grad(): _,Za=ae(xt)
Za=Za.numpy()
plt.figure(figsize=(7,5.5))
for i in range(4):
    m=(dominant==i); plt.scatter(Za[m,0],Za[m,1],s=8,c=colors[i],label=names[i],alpha=0.6)
plt.title('AE 的 2 維 latent（會彎曲，分群常比 PCA 清楚）')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4 · VAE 變分自編碼器：會「生成」的 AE

AE 的 latent 有很多「洞」——你隨便指一個座標 decode，可能得到垃圾。
**VAE** 多做兩件事，把 latent 整理成一團乾淨的雲（中心在原點）：
1. encoder 不輸出一個點，而是輸出一朵**小雲**（中心 `mu`、大小 `logvar`），從雲裡**抽一點**。
2. 加一個 **KL** 懲罰，逼所有雲都靠近原點、長得規矩。

整理乾淨後 → **隨便從原點附近抽座標 decode，就能生成「像真的但全新」的資料**。

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4.6))
a=rng.normal(size=(200,2))*np.array([3,0.4]); a=a@np.array([[1,1],[0,1]])+np.array([4,4])
ax[0].scatter(a[:,0],a[:,1],s=10,color='#4C72B0'); ax[0].set_title('AE latent：擠一邊、有洞\n隨便抽點 decode  可能是垃圾')
ax[0].scatter([0],[0],marker='*',s=200,color='red'); ax[0].annotate('想生成?\n這裡是洞',(0,0),(-3,-3),color='red')
v=rng.normal(size=(200,2)); ax[1].scatter(v[:,0],v[:,1],s=10,color='#55A868')
ax[1].scatter([0.5],[0.4],marker='*',s=200,color='red'); ax[1].annotate('抽這裡\ndecode新資料',(0.5,0.4),(1.4,1.6),color='red')
ax[1].set_title('VAE latent：被 KL 整成原點附近的雲\n隨便抽  生成新資料')
for a_ in ax: a_.grid(alpha=0.3); a_.axis('equal')
plt.tight_layout(); plt.show()

In [ ]:
import torch.nn.functional as F
torch.manual_seed(0)
class VAE(nn.Module):
    def __init__(s):
        super().__init__()
        s.body=nn.Sequential(nn.Linear(D,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU())
        s.mu=nn.Linear(16,2); s.lv=nn.Linear(16,2)
        s.dec=nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,32),nn.ReLU(),nn.Linear(32,D))
    def enc(s,x): h=s.body(x); return s.mu(h), s.lv(h)
    def forward(s,x):
        m,lv=s.enc(x); z=m+torch.exp(0.5*lv)*torch.randn_like(lv); return s.dec(z),m,lv
v=VAE(); opt=torch.optim.Adam(v.parameters(),1e-3)
for e in range(120):
    xh,m,lv=v(xt); rec=F.mse_loss(xh,xt)
    kl=(-0.5*(1+lv-m**2-lv.exp()).sum(1)).mean(); loss=rec+1.0*kl
    opt.zero_grad(); loss.backward(); opt.step()
print('VAE 還原=%.3f  KL=%.3f'%(rec.item(),kl.item()))

# 生成：從原點附近隨機抽 800 點 decode
v.eval()
with torch.no_grad():
    gen=v.dec(torch.randn(800,2)).numpy()
real2=PCA(n_components=2).fit(Xs); R=real2.transform(Xs); G=real2.transform(gen)
plt.figure(figsize=(7,5.5))
plt.scatter(R[:,0],R[:,1],s=8,c='#bbbbbb',label='真實街區',alpha=0.6)
plt.scatter(G[:,0],G[:,1],s=8,c='#C44E52',label='VAE 生成',alpha=0.5)
plt.title('VAE 生成的新街區(紅) 蓋在真實(灰)上 → 有蓋到才算學會')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5 · 三兄弟總整理

- **PCA**：最快最簡單，只會直線投影。先用它看資料長相。
- **AE**：會彎曲，壓縮/分群通常更好；但不能生成。
- **VAE**：latent 整齊 → **能生成新資料**，是 GAN/Diffusion 之前最該懂的生成模型基石。

選擇：**只想看 → PCA**；**想壓得更好 → AE**；**想造新資料 → VAE**。

概念懂了，回去把隔壁 **`1_PCA` → `2_AE` → `3_VAE`** 三本各跑一遍，
每一行程式你在第 1、2 本都學過了。